# Nền tảng 2 — Thống kê suy diễn cho các kết luận của project

Notebook này dựng lại toàn bộ bộ máy suy luận đứng sau **một dòng** trong `results/summary.md`:

> `| H1 | super-nfc − bpe-nfc | clean | +0.0018 | [+0.0012, +0.0024] | yes |`

Sau notebook này bạn giải thích được từng ô trong dòng đó, biết nó dựa trên giả định nào, và biết nó **không**
bao phủ cái gì.

**Bạn đã có**: trung bình, phương sai, độ lệch chuẩn — tức thống kê mô tả. Mọi thứ ở đây xây trên đúng ba thứ đó.

**Notebook khai triển**: mẫu và tổng thể; phân phối lấy mẫu; vì sao $\mathrm{SE} = \sigma/\sqrt{n}$; định lý giới
hạn trung tâm và nguồn gốc con số 1,96; khoảng tin cậy; kiểm định giả thuyết; hiệp phương sai và thiết kế ghép
cặp; bootstrap; McNemar; effect size và power; phân rã nguồn nhiễu; so sánh nhiều lần.

Mọi con số đều tính trực tiếp từ kết quả thật của project (d8, so `super-nfc` với `bpe-nfc`, $n = 1996$ văn bản).
Không có số nào chép tay vào — cell nào cũng chạy được, sửa được.

**Chạy bằng kernel pixi của project** (`.pixi/envs/default/bin/python`).

In [ ]:
import json
import math
import sys
from itertools import product
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:   # tìm gốc repo, chạy được từ mọi thư mục
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
RUNS = ROOT / "kaggle" / "outputs"


def doc_load(path):
    return json.loads(Path(path).read_text())


A = doc_load(RUNS / "results-v6" / "results" / "super-nfc_d8_s0.json")   # điều kiện A
B = doc_load(RUNS / "results-v6" / "results" / "bpe-nfc_d8_s0.json")     # điều kiện B
print("đã nạp:", A["user_config"]["depth"], "lớp |", "max_seq_len A =", A["max_seq_len"], "| B =", B["max_seq_len"])

## 0. Dữ liệu thô của mọi phép tính sau đây

File kết quả lưu, cho **từng văn bản test**, hai số đã gặp ở notebook 01:

- `chars` — số ký tự NFC của văn bản $i$, ký hiệu $C_i$;
- `nats` — tổng nat mà model tốn để sinh văn bản đó, ký hiệu $N_i$; bằng `None` nếu văn bản vượt context.

Mỗi điều kiện bỏ qua một số văn bản khác nhau (SuperBPE nén tốt hơn nên `max_seq_len` khác), nên trước khi so
phải lấy **giao** của các tập văn bản chấm được. Đó đúng là việc
[`vitok.analysis.common_docs`](../../../src/vitok/analysis.py) làm, và mục 7.4 sẽ giải thích vì sao bước này bắt buộc.

In [ ]:
def lay_giao(runs, variant="clean"):
    ok = np.ones(len(runs[0]["docs"][variant]["nats"]), bool)
    for r in runs:
        ok &= np.array([n is not None for n in r["docs"][variant]["nats"]])
    return np.flatnonzero(ok)


def nats_cua(run, idx, variant="clean"):
    return np.array([run["docs"][variant]["nats"][i] for i in idx], float)


idx = lay_giao([A, B])
nats_a, nats_b = nats_cua(A, idx), nats_cua(B, idx)
chars = np.array([A["docs"]["clean"]["chars"][i] for i in idx], float)
n = len(idx)

print(f"tổng số văn bản test : {len(A['docs']['clean']['nats'])}")
print(f"A chấm được          : {sum(x is not None for x in A['docs']['clean']['nats'])}")
print(f"B chấm được          : {sum(x is not None for x in B['docs']['clean']['nats'])}")
print(f"giao (n dùng để so)  : {n}")


def bpc(nats, chars):
    return float(np.sum(nats) / (math.log(2) * np.sum(chars)))


bpc_a, bpc_b = bpc(nats_a, chars), bpc(nats_b, chars)
delta_hat = bpc_a - bpc_b
print(f"\nbpc(super-nfc) = {bpc_a:.6f}")
print(f"bpc(bpe-nfc)   = {bpc_b:.6f}")
print(f"Δ̂ = A - B      = {delta_hat:+.6f} bpc   <- con số ở cột 4 của dòng H1")

## 1. Mẫu, tổng thể, và câu hỏi thật sự cần trả lời

$\hat{\Delta} = +0{,}0018$ bpc là một **sự thật về 1.996 văn bản cụ thể** trong `test.jsonl`. Nhưng phát biểu bạn
muốn đưa vào báo cáo không phải về 1.996 văn bản đó, mà về **văn bản tiếng Việt nói chung**.

Thống kê suy diễn đặt tên cho hai đối tượng:

- **Tổng thể** (population): tập mọi đối tượng ta muốn phát biểu — ở đây "mọi văn bản tiếng Việt model có thể
  gặp", vô hạn và không quan sát được trọn vẹn.
- **Mẫu** (sample): tập hữu hạn ta thực sự quan sát — 1.996 văn bản.

Giả định nền: mẫu được rút **ngẫu nhiên và độc lập** từ tổng thể. Trong project, giả định được bảo đảm ở mức chấp
nhận được vì `test.jsonl` lấy từ split `test` riêng của FineWeb-2, lọc theo hash nên không trùng dữ liệu train.
Nó không hoàn hảo — văn bản web không thực sự độc lập với nhau — và đó là một hạn chế nên ghi vào báo cáo.

Câu hỏi trung tâm của suy diễn: **nếu rút một mẫu khác, $\hat{\Delta}$ sẽ lệch đi bao nhiêu?**

Cell dưới cho thấy câu hỏi này không phải lý thuyết suông: chia đôi chính tập test hiện có, hai nửa đã cho hai
con số khác nhau.

In [ ]:
rng = np.random.default_rng(0)
hoan_vi = rng.permutation(n)
nua1, nua2 = hoan_vi[: n // 2], hoan_vi[n // 2:]

d1 = bpc(nats_a[nua1], chars[nua1]) - bpc(nats_b[nua1], chars[nua1])
d2 = bpc(nats_a[nua2], chars[nua2]) - bpc(nats_b[nua2], chars[nua2])
print(f"Δ̂ trên nửa 1 ({len(nua1)} văn bản) = {d1:+.6f}")
print(f"Δ̂ trên nửa 2 ({len(nua2)} văn bản) = {d2:+.6f}")
print(f"chênh lệch giữa hai nửa           = {abs(d1 - d2):.6f}")
print("\n=> bản thân Δ̂ là một biến ngẫu nhiên. Suy diễn là việc mô tả độ dao động đó.")

## 2. Ước lượng và phân phối lấy mẫu

**Ước lượng** (estimator) là một hàm của mẫu. Ở đây:

$$\hat{\Delta}(\text{mẫu}) = \frac{\sum_{i \in \text{mẫu}} N_i^{A}}{\ln 2 \sum_{i \in \text{mẫu}} C_i} - \frac{\sum_{i \in \text{mẫu}} N_i^{B}}{\ln 2 \sum_{i \in \text{mẫu}} C_i}$$

**Ký hiệu mới**
- $N_i^{A}$, $N_i^{B}$ — tổng nat của văn bản $i$ dưới điều kiện A, B (chỉ số trên là **tên điều kiện**, không phải số mũ)
- $\hat{\Delta}$ — dấu mũ nghĩa là **ước lượng** từ dữ liệu; $\Delta$ không mũ là giá trị thật, ta không biết

Vì mẫu là ngẫu nhiên, $\hat{\Delta}$ **là một biến ngẫu nhiên**. Phân phối của nó — tập hợp các giá trị nó nhận
được nếu ta lặp lại việc lấy mẫu vô số lần — gọi là **phân phối lấy mẫu** (sampling distribution).

Đây là chỗ dễ nhầm nhất khi mới học vì có hai tầng ngẫu nhiên chồng lên nhau:

- **Tầng dữ liệu**: từng văn bản có độ khó khác nhau (độ lệch chuẩn $\approx 0{,}24$ bpc, tính ở mục 7).
- **Tầng ước lượng**: con số tổng hợp $\hat{\Delta}$ dao động vì mẫu dao động (độ lệch chuẩn $\approx 0{,}0003$).

Mọi đại lượng suy diễn — sai số chuẩn, khoảng tin cậy, p-value — đều nói về **tầng thứ hai**, không phải tầng thứ
nhất. Hai tầng chênh nhau gần 1.000 lần, nên lẫn chúng là lẫn rất lớn.

**Ví dụ nhỏ nhất có thể liệt kê hết**: tổng thể chỉ có 4 phần tử $\{1, 2, 3, 4\}$, lấy mẫu $n = 2$ có hoàn lại.
Có $4^2 = 16$ mẫu khả dĩ. Liệt kê cả 16 và xem trung bình mẫu phân phối ra sao — đó chính là phân phối lấy mẫu,
lần này không cần mô phỏng vì đếm được hết.

In [ ]:
tong_the = [1, 2, 3, 4]
mau = list(product(tong_the, repeat=2))
trung_binh = [sum(m) / 2 for m in mau]

print(f"trung bình tổng thể μ = {np.mean(tong_the)}")
print(f"số mẫu khả dĩ         = {len(mau)}\n")
print("giá trị | số mẫu cho ra giá trị đó")
for v in sorted(set(trung_binh)):
    dem = trung_binh.count(v)
    print(f"  {v:4.1f}  | {'#' * dem} ({dem}/16)")

print(f"\nkỳ vọng của trung bình mẫu = {np.mean(trung_binh)}  (bằng đúng μ -> ước lượng không chệch)")
sigma = np.std(tong_the)                      # độ lệch chuẩn của DỮ LIỆU
se = np.std(trung_binh)                       # độ lệch chuẩn của ƯỚC LƯỢNG
print(f"σ (tầng dữ liệu)   = {sigma:.4f}")
print(f"SE (tầng ước lượng) = {se:.4f}  =  σ/√2 = {sigma / math.sqrt(2):.4f}")

## 3. Sai số chuẩn và vì sao nó có dạng $\sigma/\sqrt{n}$

**Sai số chuẩn** (standard error) là độ lệch chuẩn của phân phối lấy mẫu:

$$\mathrm{SE}(\hat{\theta}) = \sqrt{\operatorname{Var}(\hat{\theta})}$$

**Ký hiệu mới**
- $\theta$ — ký hiệu chung cho đại lượng cần ước lượng (ở project là $\Delta$); $\hat\theta$ — ước lượng của nó
- $\operatorname{Var}(\hat{\theta})$ — phương sai của $\hat\theta$ **qua các lần lấy mẫu khác nhau**

Phân biệt hai ký hiệu, vì đây là nguồn nhầm lẫn thường xuyên: $\sigma$ là độ lệch chuẩn **của dữ liệu**,
$\mathrm{SE}$ là độ lệch chuẩn **của ước lượng**. Khác nhau về ý nghĩa, và khác nhau về độ lớn theo hệ số
$\sqrt{n}$.

Dẫn ra cho trường hợp trung bình. Cho $X_1, \dots, X_n$ độc lập, cùng phân phối, phương sai $\sigma^2$. Chỉ cần
hai tính chất của phương sai mà bạn đã có:

- $\operatorname{Var}(cX) = c^2 \operatorname{Var}(X)$, với $c$ là một hằng số (nhân dữ liệu với $c$ thì độ phân tán\n  nhân với $c$, phương sai nhân với $c^2$)
- $\operatorname{Var}(X + Y) = \operatorname{Var}(X) + \operatorname{Var}(Y)$ khi $X$, $Y$ độc lập

thì

$$\operatorname{Var}(\bar{X}) = \operatorname{Var}\!\left(\frac{1}{n}\sum_{i=1}^{n} X_i\right) = \frac{1}{n^2}\sum_{i=1}^{n}\operatorname{Var}(X_i) = \frac{n\sigma^2}{n^2} = \frac{\sigma^2}{n} \quad\Rightarrow\quad \mathrm{SE}(\bar{X}) = \frac{\sigma}{\sqrt{n}}$$

**Ký hiệu mới:** $\bar{X}$ — trung bình mẫu (dấu gạch trên đầu = "trung bình của mẫu").

**Đọc từng bước:** đưa $\frac{1}{n}$ ra ngoài bằng tính chất 1, tách phương sai của tổng bằng tính chất 2, rồi
cộng $n$ số hạng bằng nhau.

Hệ quả thực tiễn: **muốn giảm sai số chuẩn một nửa phải tăng mẫu gấp 4 lần.** Trong project, tăng tập test từ
2.000 lên 4.000 văn bản chỉ giúp $\sqrt{2} \approx 1{,}41$ lần.

Thực tế ta không biết $\sigma$ nên thay bằng độ lệch chuẩn mẫu $s$; với $n = 1996$ sai khác không đáng kể.

Kiểm chứng bằng mô phỏng, dùng chính dữ liệu project: coi 1.996 hiệu bpc theo văn bản là "tổng thể", rút mẫu con
nhiều lần, rồi so độ lệch chuẩn đo được của các trung bình mẫu với $\sigma/\sqrt{n}$.

In [ ]:
# hiệu bpc theo từng văn bản: đại lượng mà thiết kế ghép cặp thực sự làm việc trên đó
hieu_theo_van_ban = (nats_a - nats_b) / (chars * math.log(2))
sigma_pop = hieu_theo_van_ban.std()
print(f"σ của 'tổng thể' 1996 hiệu = {sigma_pop:.6f}\n")

rng = np.random.default_rng(1)
print(" n    | SE mô phỏng | σ/√n       | tỷ lệ")
for m in (25, 100, 400, 1600):
    lay = rng.integers(0, n, size=(4000, m))
    tb = hieu_theo_van_ban[lay].mean(axis=1)
    se_mo_phong, se_ly_thuyet = tb.std(), sigma_pop / math.sqrt(m)
    print(f"{m:5d} | {se_mo_phong:.6f}    | {se_ly_thuyet:.6f}   | {se_mo_phong / se_ly_thuyet:.3f}")
print("\nn tăng 4 lần -> SE giảm đúng 2 lần.")

## 4. Định lý giới hạn trung tâm và nguồn gốc con số 1,96

**Định lý giới hạn trung tâm** (central limit theorem): với $X_i$ độc lập cùng phân phối, kỳ vọng $\mu$, phương
sai $\sigma^2$ hữu hạn, thì khi $n$ lớn

$$\frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \ \xrightarrow{\ d\ } \ \mathcal{N}(0, 1)$$

**Ký hiệu mới**
- $\mu$ — kỳ vọng thật của một quan sát
- $\mathcal{N}(0, 1)$ — phân phối chuẩn chuẩn tắc (kỳ vọng 0, phương sai 1)
- $\xrightarrow{\ d\ }$ — "hội tụ theo phân phối": khi $n \to \infty$, phân phối vế trái ngày càng giống vế phải

Ý nghĩa: **bất kể phân phối gốc có hình dạng gì**, phân phối lấy mẫu của trung bình tiến về phân phối chuẩn. Đây
là lý do phân phối chuẩn xuất hiện khắp nơi trong thống kê ứng dụng dù dữ liệu gốc không chuẩn. Trong project,
bpc theo từng văn bản lệch phải rõ rệt (vài văn bản rất khó, kéo đuôi bên phải), nhưng trung bình trên 1.996 văn
bản thì gần chuẩn.

Đại lượng đo độ lệch của một phân phối là **độ xiên** (skewness):

$$\mathrm{skew}(X) = \mathbb{E}\!\left[\left(\frac{X - \mu}{\sigma}\right)^{3}\right]$$

Số hạng trong ngoặc là giá trị đã **chuẩn hoá** (không còn đơn vị); luỹ thừa 3 giữ dấu nên đuôi phải cho đóng
góp dương, đuôi trái cho đóng góp âm.

Bằng 0 với phân phối đối xứng, dương khi đuôi phải dài. Cell dưới đo độ xiên của dữ liệu gốc rồi đo lại của trung
bình mẫu, để thấy CLT "kéo" phân phối về đối xứng khi $n$ tăng.

In [ ]:
def do_xien(x):
    x = np.asarray(x, float)
    return float((((x - x.mean()) / x.std()) ** 3).mean())


bpc_goc = nats_a / (chars * math.log(2))      # bpc của từng văn bản, điều kiện A
print(f"độ xiên của bpc theo văn bản (dữ liệu gốc): {do_xien(bpc_goc):+.3f}")

rng = np.random.default_rng(2)
for m in (1, 5, 30, 200):
    tb = bpc_goc[rng.integers(0, n, size=(20000, m))].mean(axis=1)
    print(f"  độ xiên của trung bình mẫu, n={m:4d}: {do_xien(tb):+.3f}")
print("\nn càng lớn, độ xiên càng về 0: đó là CLT đang làm việc.")

Con số **1,96** đến từ đây. Với $Z \sim \mathcal{N}(0,1)$, ta tìm $z$ sao cho $P(-z \le Z \le z) = 0{,}95$. Nghiệm
là **phân vị** (quantile) mức $0{,}975$ của phân phối chuẩn chuẩn tắc:

$$z_{0{,}975} = 1{,}959964 \approx 1{,}96$$

**Ký hiệu mới:** $z_q$ — **phân vị mức $q$** của $\mathcal{N}(0,1)$, tức số thoả $P(Z \le z_q) = q$.

Vì sao mức $0{,}975$ chứ không phải $0{,}95$: cần chừa $2{,}5\%$ ở **mỗi** đuôi, tổng $5\%$, nên biên phải nằm ở
mức tích luỹ $1 - 0{,}05/2$.

Hàm phân phối tích luỹ của chuẩn chuẩn tắc viết được bằng hàm sai số `math.erf` có sẵn:

$$\Phi(z) = P(Z \le z) = \frac{1}{2}\left(1 + \operatorname{erf}\!\left(\frac{z}{\sqrt{2}}\right)\right)$$

**Ký hiệu mới**
- $\Phi(z)$ — hàm phân phối tích luỹ của $\mathcal{N}(0,1)$
- $\operatorname{erf}$ — hàm sai số, có sẵn là `math.erf`

Không có hàm ngược dạng đóng, nhưng $\Phi$ tăng nghiêm ngặt nên chia đôi khoảng (bisection) là đủ: mỗi bước cắt
đôi khoảng chứa nghiệm, sau 60 bước sai số dưới $10^{-15}$.

In [ ]:
def Phi(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2)))


def phan_vi_chuan(p, lo=-10.0, hi=10.0):
    for _ in range(200):
        mid = (lo + hi) / 2
        if Phi(mid) < p:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


for muc, ten in [(0.95, "CI 90%"), (0.975, "CI 95%"), (0.995, "CI 99%")]:
    print(f"z_{{{muc}}} = {phan_vi_chuan(muc):.6f}   ({ten})")

z95 = phan_vi_chuan(0.975)
print(f"\nkiểm tra: P(-z ≤ Z ≤ z) với z = {z95:.4f} là {Phi(z95) - Phi(-z95):.6f}")

## 5. Khoảng tin cậy

### 5.1. Định nghĩa

Khoảng tin cậy mức $1-\alpha$ không phải một khoảng cố định, mà là một **quy tắc** dựng khoảng từ dữ liệu sao cho

$$P\big(\text{khoảng dựng được chứa } \theta\big) = 1 - \alpha$$

**Ký hiệu mới:** $\alpha$ — xác suất cho phép khoảng trượt khỏi giá trị thật; $1-\alpha$ là **mức tin cậy**
($\alpha = 0{,}05$ cho khoảng 95%).

trong đó xác suất lấy theo việc **lặp lại toàn bộ thí nghiệm**. Khi phân phối lấy mẫu xấp xỉ chuẩn:

$$\mathrm{CI}_{1-\alpha} = \hat{\theta} \pm z_{1-\alpha/2} \cdot \mathrm{SE}(\hat{\theta})$$

### 5.2. Cách hiểu đúng và cách hiểu sai

Cách hiểu **sai** thường gặp: "có 95% xác suất giá trị thật nằm trong khoảng này".

Vì sao sai: trong khung tần suất (frequentist), $\theta$ là hằng số chưa biết, không phải biến ngẫu nhiên. Nó nằm
trong khoảng hoặc không — xác suất là 0 hoặc 1, chỉ là ta không biết trường hợp nào. Cái ngẫu nhiên là **khoảng**,
vì khoảng phụ thuộc mẫu.

Cách hiểu **đúng**: nếu lặp lại thí nghiệm 100 lần, mỗi lần dựng một khoảng theo cùng quy tắc, thì khoảng 95
trong số 100 khoảng đó chứa $\theta$.

Phát biểu này kiểm chứng trực tiếp được, và cell dưới làm đúng thế: coi 1.996 văn bản là tổng thể (nên $\theta$
biết trước), rút 5.000 mẫu con, dựng 5.000 khoảng, rồi đếm bao nhiêu khoảng chứa $\theta$.

Muốn phát biểu dạng "xác suất $\theta$ nằm trong khoảng" thì cần khung Bayes và một phân phối tiên nghiệm, cho ra
*credible interval* — khái niệm khác, không dùng trong project này.

In [ ]:
theta = hieu_theo_van_ban.mean()      # "giá trị thật" trong thế giới mô phỏng này
rng = np.random.default_rng(3)
m, R = 200, 5000
lay = rng.integers(0, n, size=(R, m))
mau = hieu_theo_van_ban[lay]
tb, s = mau.mean(axis=1), mau.std(axis=1, ddof=1)
lo, hi = tb - z95 * s / math.sqrt(m), tb + z95 * s / math.sqrt(m)
chua = (lo <= theta) & (theta <= hi)

print(f"θ (biết trước)        = {theta:+.6f}")
print(f"số khoảng chứa θ      = {chua.sum()}/{R} = {chua.mean():.1%}   (kỳ vọng 95%)")
print(f"độ rộng khoảng trung bình = {(hi - lo).mean():.6f}\n")
print("năm khoảng đầu tiên:")
for i in range(5):
    print(f"  [{lo[i]:+.6f}, {hi[i]:+.6f}]  {'chứa θ' if chua[i] else 'TRƯỢT'}")

### 5.3. Điều khoảng tin cậy không bao phủ

Công thức chỉ chứa **một** nguồn ngẫu nhiên: việc chọn mẫu văn bản nào. Mọi nguồn ngẫu nhiên khác của thí nghiệm
— khởi tạo trọng số, thứ tự dữ liệu, tính bất định của số học fp16 — không có mặt trong đó. Đây là hạn chế quan
trọng nhất của bảng kết quả, và mục 11 xử lý riêng.

## 6. Kiểm định giả thuyết

### 6.1. Bốn thành phần

1. **Giả thuyết không** $H_0$: phát biểu mặc định, thường là "không có hiệu ứng", ở đây $H_0: \Delta = 0$. Bên
   cạnh là **giả thuyết đối** $H_1: \Delta \neq 0$ (hai phía).
2. **Thống kê kiểm định**: một số tính từ dữ liệu mà ta biết phân phối của nó **khi $H_0$ đúng**. Ở đây
   $z = \hat{\Delta}/\mathrm{SE}$, và khi $H_0$ đúng thì $z \sim \mathcal{N}(0,1)$ theo mục 4. Phân phối này gọi
   là **phân phối null**.
3. **p-value**: xác suất, **với điều kiện $H_0$ đúng**, quan sát được thống kê cực đoan bằng hoặc hơn giá trị
   thực tế:

$$p = P\big(|Z| \ge |z_{\text{quan sát}}| \ \big|\ H_0\big) = 2\big(1 - \Phi(|z_{\text{quan sát}}|)\big)$$

   **Ký hiệu mới:** $z_{\text{quan sát}}$ — giá trị $z$ tính từ dữ liệu thật. Trị tuyệt đối vì "cực đoan" tính cả
   hai phía; nhân 2 để cộng diện tích hai đuôi.

4. **Mức ý nghĩa** $\alpha$: ngưỡng chọn trước, thường 0,05. Nếu $p < \alpha$ thì "bác bỏ $H_0$".

### 6.2. Đối ngẫu giữa khoảng tin cậy và kiểm định

Hai công cụ này là một, nhìn từ hai phía:

$$\text{CI}_{95\%} \text{ không chứa } 0 \quad \Longleftrightarrow \quad p < 0{,}05$$

Vì $\mathrm{CI}$ không chứa 0 nghĩa là $|\hat\Delta| > 1{,}96\,\mathrm{SE}$, tức $|z| > 1{,}96$, tức $p < 0{,}05$
— cùng một bất đẳng thức viết ba kiểu. Đối ngẫu này **chính xác** khi khoảng tin cậy dựng bằng công thức
$\hat\Delta \pm 1{,}96\,\mathrm{SE}$ và kiểm định dùng cùng $\mathrm{SE}$ đó. Project dựng khoảng bằng **phân vị
bootstrap** (mục 8), nên đối ngẫu chỉ còn là xấp xỉ — rất sát ở đây vì phân phối bootstrap gần đối xứng. Cột
Cột `CI excludes 0` trong `results/summary.md` (khoá `significant`) được cài thẳng bằng điều kiện khoảng, xem [`vitok.stats.paired_bootstrap_bpc`](../../../src/vitok/stats.py):

```python
"significant": bool(lo > 0 or hi < 0)
```

**Một lưu ý về $\mathrm{SE}$ dùng trong cell dưới.** $\hat\Delta$ là hiệu của hai *tỷ lệ giữa hai tổng*, không
phải một trung bình, nên không có công thức $\sigma/\sqrt n$ áp thẳng được. Cách xấp xỉ nhanh: coi mẫu số
$\ln 2 \sum_i C_i$ là cố định, chỉ tử số dao động, thì

$$\mathrm{SE}(\hat\Delta) \approx \frac{\sqrt{n}\ \mathrm{sd}(N_i^A - N_i^B)}{\ln 2 \sum_i C_i} = \frac{\mathrm{sd}(N_i^A - N_i^B)}{\ln 2\ \bar{C}\ \sqrt{n}}$$

**Ký hiệu mới:** $\mathrm{sd}(\cdot)$ — độ lệch chuẩn mẫu của dãy trong ngoặc; $\bar C$ — số ký tự trung bình
mỗi văn bản.

Đây là **phương pháp delta** ở dạng đơn giản nhất. Nó cho mỗi văn bản trọng số tỷ lệ với số ký tự của văn bản
đó — đúng như định nghĩa bpc. Mục 8 tính lại bằng bootstrap, không cần xấp xỉ nào: hai cách cho $0{,}000307$ và
$0{,}000309$, lệch nhau dưới 1%.

In [ ]:
se_ghep = (nats_a - nats_b).std(ddof=1) / (chars.mean() * math.log(2) * math.sqrt(n))
z = delta_hat / se_ghep
p = 2 * (1 - Phi(abs(z)))
print(f"Δ̂  = {delta_hat:+.6f}")
print(f"SE = {se_ghep:.6f}")
print(f"z  = Δ̂/SE = {z:.2f}")
print(f"p  = 2(1 - Φ(|z|)) = {p:.3e}\n")

print("đối ngẫu CI <-> p, quét qua vài giá trị hiệu ứng giả định:")
print(" Δ giả định | CI 95%                   | chứa 0 | p       | p<0.05")
for d in (0.0002, 0.0005, 0.0007, 0.0018):
    l, h = d - z95 * se_ghep, d + z95 * se_ghep
    pv = 2 * (1 - Phi(abs(d / se_ghep)))
    print(f"  {d:+.4f}   | [{l:+.6f}, {h:+.6f}] | {str(l <= 0 <= h):5s}  | {pv:7.4f} | {str(pv < 0.05):5s}")
print("\nhai cột cuối luôn ngược nhau: đó chính là đối ngẫu.")

### 6.3. Hai loại sai lầm và power

| | $H_0$ đúng | $H_0$ sai |
|---|---|---|
| Bác bỏ $H_0$ | **Sai lầm loại I**, xác suất $\alpha$ | Quyết định đúng, xác suất $1-\beta$ |
| Không bác bỏ | Quyết định đúng | **Sai lầm loại II**, xác suất $\beta$ |

**Power** $= 1 - \beta$ là xác suất phát hiện được hiệu ứng khi nó thật sự tồn tại. Power phụ thuộc bốn thứ: độ
lớn hiệu ứng thật, $n$, độ biến thiên của dữ liệu, và $\alpha$. Mục 10 tính power cho thiết kế của bạn.

Cell dưới mô phỏng cả hai loại sai lầm: sinh dữ liệu từ một thế giới có $H_0$ đúng (tỷ lệ bác bỏ phải ra $\approx
\alpha$), rồi từ một thế giới có hiệu ứng thật (tỷ lệ bác bỏ chính là power).

In [ ]:
rng = np.random.default_rng(4)
R, m, sd = 20000, 200, 1.0


def ty_le_bac_bo(hieu_ung_that):
    x = rng.normal(hieu_ung_that, sd, size=(R, m))
    z_stat = x.mean(axis=1) / (x.std(axis=1, ddof=1) / math.sqrt(m))
    return float((np.abs(z_stat) > z95).mean())


print(f"H0 đúng (Δ = 0)      -> bác bỏ {ty_le_bac_bo(0.0):.3%}   (kỳ vọng α = 5%: sai lầm loại I)")
for d in (0.10, 0.20, 0.30):
    pw = ty_le_bac_bo(d)
    print(f"H0 sai  (Δ = {d:.2f})   -> bác bỏ {pw:.1%}   power = {pw:.1%}, sai lầm loại II = {1 - pw:.1%}")

### 6.4. Ba cách hiểu sai p-value

- **p-value không phải $P(H_0 \mid \text{dữ liệu})$.** Nó là xác suất của dữ liệu khi giả sử $H_0$, tức chiều
  điều kiện ngược lại. Đổi chiều điều kiện là một sai lầm logic riêng, có tên là *nguỵ biện công tố viên*
  (prosecutor's fallacy).
- **$p < 0{,}05$ không nói hiệu ứng lớn.** Xem mục 10.
- **$p > 0{,}05$ không chứng minh $H_0$ đúng.** Nó chỉ nói dữ liệu không đủ để bác bỏ — có thể vì hiệu ứng không
  tồn tại, mà cũng có thể vì power thấp. Trường hợp cặp tối thiểu ở mục 9 đúng là khả năng thứ hai.

## 7. Thiết kế ghép cặp

### 7.1. Hiệp phương sai và tương quan

Hai đại lượng cần có trước khi nói về ghép cặp.

**Hiệp phương sai** (covariance) đo mức độ hai biến cùng lệch khỏi kỳ vọng của chúng:

$$\operatorname{Cov}(X, Y) = \mathbb{E}\big[(X - \mathbb{E}X)(Y - \mathbb{E}Y)\big]$$

**Ký hiệu mới:** $X$, $Y$ — hai phép đo trên **cùng một** đối tượng (ở project: bpc của cùng một văn bản dưới hai
model).

Dương khi hai biến có xu hướng cùng lớn hoặc cùng nhỏ; bằng 0 khi không có quan hệ tuyến tính.

**Hệ số tương quan** (correlation) là hiệp phương sai chuẩn hoá về $[-1, 1]$:

$$\rho = \frac{\operatorname{Cov}(X, Y)}{\sqrt{\operatorname{Var}(X)\operatorname{Var}(Y)}}$$

### 7.2. Phương sai của hiệu

$$\operatorname{Var}(X - Y) = \operatorname{Var}(X) + \operatorname{Var}(Y) - 2\operatorname{Cov}(X, Y)$$

Đây là chìa khoá của cả thiết kế. Nếu $X$ và $Y$ tương quan mạnh ($\rho$ gần 1), số hạng
$-2\operatorname{Cov}$ triệt tiêu gần hết hai số hạng đầu, và phương sai của hiệu nhỏ hơn hẳn phương sai của từng
biến.

### 7.3. Áp dụng vào project

Hai model được chấm trên **cùng một tập văn bản**, nên ta có từng cặp $(a_i, b_i)$ — bpc của văn bản $i$ dưới
điều kiện A và B. Văn bản khó thì khó với cả hai model, nên hai dãy tương quan rất mạnh. Hai cách tính sai số
chuẩn:

| Cách | Công thức |
|---|---|
| Không ghép cặp (coi hai mẫu độc lập) | $\sqrt{s_A^2/n + s_B^2/n}$ |
| Ghép cặp (lấy hiệu từng văn bản rồi mới tổng hợp) | $s_{A-B}/\sqrt{n}$ |

và tỷ lệ giữa chúng phải khớp với công thức ở 7.2:

$$\frac{\mathrm{SE}_{\text{không ghép}}}{\mathrm{SE}_{\text{ghép}}} = \sqrt{\frac{s_A^2 + s_B^2}{s_A^2 + s_B^2 - 2\rho s_A s_B}}$$

**Ký hiệu mới** (dùng cho cả bảng trên)
- $s_A$, $s_B$ — độ lệch chuẩn **mẫu** của bpc theo văn bản dưới A, B (chữ $s$ vì tính từ dữ liệu, khác $\sigma$)
- $s_{A-B}$ — độ lệch chuẩn mẫu của hiệu bpc $a_i - b_i$

Khi $s_A \approx s_B$, biểu thức rút gọn thành $1/\sqrt{1-\rho}$.

In [ ]:
a = nats_a / (chars * math.log(2))     # bpc từng văn bản, điều kiện A
b = nats_b / (chars * math.log(2))
sa, sb = a.std(ddof=1), b.std(ddof=1)
cov = float(np.cov(a, b, ddof=1)[0, 1])
rho = cov / (sa * sb)
print(f"s_A = {sa:.4f} | s_B = {sb:.4f} | Cov = {cov:.6f} | ρ = {rho:.5f}\n")

se_khong_ghep = math.sqrt(sa ** 2 / n + sb ** 2 / n)
se_ghep_cap = (a - b).std(ddof=1) / math.sqrt(n)
print(f"SE không ghép cặp = {se_khong_ghep:.6f}")
print(f"SE ghép cặp       = {se_ghep_cap:.6f}")
print(f"tỷ lệ đo được     = {se_khong_ghep / se_ghep_cap:.2f} lần")

ly_thuyet = math.sqrt((sa ** 2 + sb ** 2) / (sa ** 2 + sb ** 2 - 2 * rho * sa * sb))
print(f"tỷ lệ lý thuyết   = {ly_thuyet:.2f} lần   (công thức Var(X-Y))")
print(f"xấp xỉ 1/√(1-ρ)   = {1 / math.sqrt(1 - rho):.2f} lần\n")

print(f"nếu KHÔNG ghép cặp: z = {delta_hat / se_khong_ghep:.2f} -> p = {2 * (1 - Phi(abs(delta_hat / se_khong_ghep))):.3f}")
print(f"có ghép cặp       : z = {delta_hat / se_ghep_cap:.2f} -> p = {2 * (1 - Phi(abs(delta_hat / se_ghep_cap))):.3e}")
print("\nPhần lớn năng lực phát hiện của thiết kế đến từ ghép cặp, không phải từ kích thước mẫu.")

Một chi tiết dễ gây bối rối nếu đối chiếu hai mục: $\mathrm{SE}$ ghép cặp ở đây là $0{,}000344$, còn xấp xỉ delta
ở mục 6 (và bootstrap ở mục 8) cho $0{,}000309$. Hai con số tính trên hai thống kê hơi khác nhau:

- $s_{A-B}/\sqrt{n}$ là $\mathrm{SE}$ của **trung bình không trọng số** các hiệu bpc theo văn bản — mỗi văn bản
  một phiếu như nhau;
- $\hat\Delta$ của project là **hiệu hai tỷ lệ giữa hai tổng** — mỗi văn bản có trọng số tỷ lệ với số ký tự.

Trọng số theo độ dài làm giảm ảnh hưởng của các văn bản ngắn, vốn là các văn bản có bpc dao động mạnh nhất, nên
$\mathrm{SE}$ của $\hat\Delta$ nhỏ hơn một chút. Ở mục này điều đó không quan trọng: cả hai vế của tỷ lệ 22 lần
đều dùng cùng một quy ước, nên tỷ lệ vẫn đúng. Khi cần con số $\mathrm{SE}$ để đưa vào báo cáo thì dùng bootstrap
ở mục 8, vì nó tính đúng trên thống kê thật sự được báo cáo.

### 7.4. Điều kiện để ghép cặp hợp lệ

Hai điều kiện, cả hai đều được project bảo đảm:

1. **Cùng đối tượng**: hai model phải được chấm trên đúng cùng tập văn bản. Vì mỗi điều kiện bỏ qua một số văn
   bản vượt context khác nhau, [`vitok.analysis.common_docs`](../../../src/vitok/analysis.py) lấy **giao** trước khi
   so — đó là bước `lay_giao` ở mục 0.
2. **Giữ ghép cặp trong mọi bước tính**, kể cả khi lấy mẫu lại ở bootstrap (mục 8).

Cell dưới cho thấy điều kiện 1 không phải hình thức: nếu so bpc tính trên hai tập văn bản khác nhau, con số lệch
ngay ở chữ số có nghĩa đầu tiên của hiệu ứng.

In [ ]:
rieng_a = [i for i, x in enumerate(A["docs"]["clean"]["nats"]) if x is not None]
rieng_b = [i for i, x in enumerate(B["docs"]["clean"]["nats"]) if x is not None]
ch = A["docs"]["clean"]["chars"]

bpc_a_rieng = bpc([A["docs"]["clean"]["nats"][i] for i in rieng_a], [ch[i] for i in rieng_a])
bpc_b_rieng = bpc([B["docs"]["clean"]["nats"][i] for i in rieng_b], [ch[i] for i in rieng_b])

print(f"A chấm {len(rieng_a)} văn bản, B chấm {len(rieng_b)}; chỉ {len(set(rieng_a) ^ set(rieng_b))} văn bản lệch nhau")
print(f"Δ nếu mỗi bên dùng tập riêng = {bpc_a_rieng - bpc_b_rieng:+.6f}")
print(f"Δ trên tập giao (đúng)       = {delta_hat:+.6f}")
print(f"sai lệch                     = {abs((bpc_a_rieng - bpc_b_rieng) - delta_hat):.6f}"
      f"  ({abs((bpc_a_rieng - bpc_b_rieng) - delta_hat) / abs(delta_hat):.0%} độ lớn hiệu ứng)")

### 7.5. Một phép so không ghép cặp ngay trong project: bpb của nanochat

Trong lúc train, nanochat tự báo "Validation bpb" trên shard val. Theo con số đó, `super-nfc` **tốt hơn**
`bpe-nfc` ở cả ba cỡ; theo bpc trên tập test thì nó **kém hơn** ở d8 và d10. Hai con số trái dấu, nên phải có một
con số sai.

Nguyên nhân nằm ở thiết kế, không ở dữ liệu. nanochat chấm một **số token cố định** (2 triệu). Tokenizer SuperBPE
nén tốt hơn, nên 2 triệu token của nó phủ nhiều hơn khoảng 22% văn bản, tức là **những văn bản khác**, và các
văn bản còn bị xếp chung một hàng. Phép so đó vi phạm đúng điều kiện ở mục 7.4: hai bên không được chấm trên cùng
một tập, nên chênh lệch lẫn cả "tokenizer nào tốt hơn" lẫn "tập văn bản nào dễ hơn".

Kế hoạch phân tích có sẵn phép kiểm tra đúng: chấm lại shard val **theo từng văn bản, trên cùng một tập** cho mọi
run (`results/val/`). Cell dưới đặt ba con số cạnh nhau.

In [ ]:
from vitok.analysis import common_docs, load, nanochat_val_bpb

val_bpb = nanochat_val_bpb(RUNS)                     # số nanochat in trong train.log, không ghép cặp
test_runs, val_runs = load(RUNS), load(ROOT / "results" / "val")


def lech_ghep_cap(runs, depth):
    ids = common_docs([r for (c, dd, s), r in runs.items() if dd == depth and s == 0], "clean")
    a, b = runs[("super-nfc", depth, 0)], runs[("bpe-nfc", depth, 0)]
    return nats_cua(a, ids).sum() / nats_cua(b, ids).sum() - 1   # cùng mẫu số ký tự, nên tỷ số nats = tỷ số bpc


print(" cỡ  | bpb nanochat (không ghép cặp) | bpc val (ghép cặp) | bpc test (ghép cặp)")
for depth in (6, 8, 10):
    kg = val_bpb[("super-nfc", depth, 0)] / val_bpb[("bpe-nfc", depth, 0)] - 1
    print(f" d{depth:<3d}| {kg:+29.2%} | {lech_ghep_cap(val_runs, depth):+18.2%} | {lech_ghep_cap(test_runs, depth):+18.2%}")
print("\nHai cột ghép cặp khớp nhau ở mọi cỡ, dù một cột lấy văn bản từ phần train, một cột từ phần test của FineWeb-2.")
print("Cột không ghép cặp lệch khỏi cả hai và trái dấu ở d8, d10: cùng dữ liệu, chỉ khác thiết kế phép so.")

## 8. Bootstrap

### 8.1. Vì sao cần

Mục 3 cho công thức $\mathrm{SE}$ của **trung bình**. Nhưng $\hat{\Delta}$ không phải trung bình: nó là **hiệu
của hai tỷ lệ giữa hai tổng** (tổng nat chia tổng ký tự). Với dạng này, công thức giải tích cho phương sai đòi
hỏi khai triển xấp xỉ (phương pháp delta) và vẫn chỉ là xấp xỉ.

Bootstrap tránh hoàn toàn vấn đề đó bằng cách **mô phỏng** phân phối lấy mẫu thay vì suy ra nó bằng giải tích.

### 8.2. Nguyên lý thế chỗ

Ta muốn biết $\hat{\Delta}$ dao động ra sao khi lấy mẫu mới từ tổng thể $F$. Nhưng $F$ không có. Ý tưởng của
Efron (1979): thay $F$ bằng **phân phối thực nghiệm** $\hat{F}$ — phân phối gán xác suất $1/n$ cho mỗi văn bản
trong mẫu hiện tại.

$$\text{Thế giới thật: } F \to \text{mẫu} \to \hat{\Delta} \qquad\Longleftrightarrow\qquad \text{Thế giới bootstrap: } \hat{F} \to \text{mẫu}^* \to \hat{\Delta}^*$$

**Ký hiệu mới:** dấu sao $^*$ — đánh dấu những gì thuộc **thế giới bootstrap** (mẫu rút lại, thống kê tính trên
mẫu rút lại).

Lấy mẫu từ $\hat{F}$ chính là **rút có hoàn lại** từ mẫu hiện tại. Lập luận biện minh: nếu $\hat{F}$ đủ gần $F$
(đúng khi $n$ lớn), thì dao động của $\hat{\Delta}^*$ quanh $\hat{\Delta}$ xấp xỉ dao động của $\hat{\Delta}$
quanh $\Delta$.

### 8.3. Thuật toán

1. Rút $n$ chỉ số văn bản **có hoàn lại** từ $\{1, \dots, n\}$. Một văn bản có thể xuất hiện nhiều lần, văn bản
   khác không xuất hiện lần nào.
2. Tính lại thống kê trên mẫu đó: $\hat{\Delta}^{*(b)}$ — dấu sao nghĩa là "tính trên mẫu bootstrap", chỉ số $(b)$\n   là số thứ tự của lượt lặp.
3. Lặp $B$ lần; project dùng $B = 10\,000$. (Ở mục này $B$ là **số lượt bootstrap**, không phải điều kiện B.)
4. Ước lượng sai số chuẩn: $\mathrm{SE} \approx \mathrm{sd}\big(\hat{\Delta}^{*(1)}, \dots, \hat{\Delta}^{*(B)}\big)$.
5. **Khoảng tin cậy dạng phân vị**: lấy phân vị 2,5% và 97,5% của tập giá trị bootstrap.

Điểm then chốt về ghép cặp: mỗi lần rút, **cùng một bộ chỉ số được áp cho cả hai điều kiện** — cùng mảng `idx` —
nên tương quan $\rho = 0{,}998$ được giữ nguyên trong mô phỏng.

### 8.4. Kiểm chứng trên dữ liệu thật

Nếu phân phối bootstrap xấp xỉ chuẩn thì khoảng phân vị phải gần trùng $\hat{\Delta} \pm 1{,}96\,\mathrm{SE}$.
Cell dưới cài bootstrap từ đầu, so hai cách dựng khoảng, rồi đối chiếu với hàm của project.

In [ ]:
def bootstrap_ghep_cap(nats_a, nats_b, chars, B=10_000, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for start in range(0, B, 500):                                   # chia lô để giới hạn bộ nhớ
        lay = rng.integers(0, len(chars), size=(min(500, B - start), len(chars)))
        mau_chars = chars[lay].sum(axis=1) * math.log(2)             # CÙNG một lay cho cả A và B
        out.append((nats_a[lay].sum(axis=1) - nats_b[lay].sum(axis=1)) / mau_chars)
    return np.concatenate(out)


boot = bootstrap_ghep_cap(nats_a, nats_b, chars)
lo_b, mid_b, hi_b = np.percentile(boot, [2.5, 50, 97.5])
se_boot = boot.std()

print(f"B                     = {len(boot)}")
print(f"SE bootstrap          = {se_boot:.6f}")
print(f"khoảng phân vị 95%    = [{lo_b:+.6f}, {hi_b:+.6f}]")
print(f"Δ̂ ± 1,96·SE           = [{delta_hat - z95 * se_boot:+.6f}, {delta_hat + z95 * se_boot:+.6f}]")
print(f"độ xiên phân phối boot = {do_xien(boot):+.3f}  (gần 0 -> hai cách trên trùng nhau)\n")

from vitok.stats import paired_bootstrap_bpc
print("đối chiếu với vitok.stats.paired_bootstrap_bpc:")
print(paired_bootstrap_bpc(nats_a, nats_b, chars))

Phiên bản **sai** của cùng thuật toán: rút độc lập hai bộ chỉ số cho A và B. Nó phá tương quan $\rho = 0{,}998$,
nên mô phỏng một phân phối lấy mẫu không tương ứng với thí nghiệm đã chạy, và cho khoảng rộng gấp hơn hai chục
lần. Chạy để thấy hậu quả cụ thể.

In [ ]:
rng = np.random.default_rng(0)
sai = []
for _ in range(20):
    la = rng.integers(0, n, size=(500, n))
    lb = rng.integers(0, n, size=(500, n))                            # bộ chỉ số KHÁC -> mất ghép cặp
    sai.append(nats_a[la].sum(axis=1) / (chars[la].sum(axis=1) * math.log(2))
               - nats_b[lb].sum(axis=1) / (chars[lb].sum(axis=1) * math.log(2)))
sai = np.concatenate(sai)
lo_s, hi_s = np.percentile(sai, [2.5, 97.5])

print(f"bootstrap ghép cặp (đúng) : [{lo_b:+.6f}, {hi_b:+.6f}]  rộng {hi_b - lo_b:.6f}  -> kết luận được")
print(f"bootstrap độc lập  (sai)  : [{lo_s:+.6f}, {hi_s:+.6f}]  rộng {hi_s - lo_s:.6f}  -> chứa 0, không kết luận được")
print(f"tỷ lệ độ rộng             : {(hi_s - lo_s) / (hi_b - lo_b):.1f} lần")

### 8.5. Giới hạn của bootstrap

Bootstrap chỉ mô phỏng **việc chọn lại mẫu từ cùng một tổng thể**. Nó không biết gì về:

- ngẫu nhiên của quá trình huấn luyện (khởi tạo, thứ tự dữ liệu, fp16) — mục 11;
- sai lệch hệ thống của tập test: nếu `test.jsonl` không đại diện cho tiếng Việt, bootstrap không phát hiện được;
- phụ thuộc giữa các văn bản: bootstrap giả định độc lập.

## 9. Kiểm định McNemar cho kết quả nhị phân ghép cặp

### 9.1. Bài toán

Với cặp tối thiểu, mỗi cặp câu cho một kết quả nhị phân cho mỗi model: model gán ít nat hơn cho câu đúng ngữ pháp
thì tính là **đúng**. Lập bảng $2\times2$ đếm số cặp theo bốn tổ hợp:

| | B đúng | B sai |
|---|---|---|
| **A đúng** | $n_{11}$ | $n_{10}$ |
| **A sai** | $n_{01}$ | $n_{00}$ |

### 9.2. Vì sao chỉ hai ô bất đồng mang thông tin

Các cặp mà **cả hai model cùng đúng** hoặc **cùng sai** không phân biệt được A với B: chúng nói về độ khó của cặp
câu, không nói ai hơn ai. Chỉ các cặp **bất đồng** ($n_{10}$ và $n_{01}$) mang thông tin về hướng chênh lệch.

Lập luận hình thức: đặt $m = n_{10} + n_{01}$. Điều kiện hoá theo $m$ (coi $m$ là đã biết), giả thuyết "hai model
tương đương" có nghĩa mỗi cặp bất đồng nghiêng về A hay về B với xác suất bằng nhau, nên

$$n_{10} \mid m,\ H_0 \ \sim\ \mathrm{Binomial}\!\left(m, \tfrac{1}{2}\right)$$

**Ký hiệu mới:** $\sim$ — "có phân phối"; $\mathrm{Binomial}(m, \frac12)$ — số mặt ngửa khi tung $m$ đồng xu
cân đối.

Đây chính là một phép kiểm định dấu (sign test) trên các cặp bất đồng.

### 9.3. Công thức p-value chính xác

Với $k = \min(n_{10}, n_{01})$, p-value hai phía:

$$p = \min\left(1,\ 2\sum_{i=0}^{k} \binom{m}{i} 2^{-m}\right)$$

**Ký hiệu mới:** $\binom{m}{i} = \frac{m!}{i!\,(m-i)!}$ — số cách chọn $i$ trong $m$ cặp; nhân với $2^{-m}$ ra
xác suất nhị thức $P(n_{10} = i)$.

Hệ số 2 vì hai phía; hàm $\min$ vì tổng nhân đôi có thể vượt 1 khi $n_{10} \approx n_{01}$.

Bản "chính xác" (exact) dùng trực tiếp phân phối nhị thức, khác bản xấp xỉ khi-bình-phương thường gặp. Với $m$
nhỏ — đúng tình huống của project — bản xấp xỉ không đáng tin, nên
[`vitok.stats.mcnemar_exact`](../../../src/vitok/stats.py) cài bản chính xác.

In [ ]:
dung_a = np.array(A["pairs"]["good_nats"]) < np.array(A["pairs"]["bad_nats"])
dung_b = np.array(B["pairs"]["good_nats"]) < np.array(B["pairs"]["bad_nats"])

n11 = int((dung_a & dung_b).sum())
n10 = int((dung_a & ~dung_b).sum())
n01 = int((~dung_a & dung_b).sum())
n00 = int((~dung_a & ~dung_b).sum())
print(f"số cặp tối thiểu = {len(dung_a)}")
print(f"acc A = {dung_a.mean():.4f} | acc B = {dung_b.mean():.4f}\n")
print("            B đúng   B sai")
print(f"A đúng      {n11:6d}  {n10:6d}")
print(f"A sai       {n01:6d}  {n00:6d}")

m_bd, k = n10 + n01, min(n10, n01)
p_mc = min(1.0, 2 * sum(math.comb(m_bd, i) for i in range(k + 1)) / 2 ** m_bd)
print(f"\nm = {m_bd} cặp bất đồng, k = {k}")
print(f"p (exact, hai phía) = {p_mc:.4f}")

from vitok.stats import mcnemar_exact
print("đối chiếu vitok.stats.mcnemar_exact:", mcnemar_exact(dung_a, dung_b))

### 9.4. Số của project và hiệu ứng chạm trần

Vì sao phép đo này không kết luận được gì: độ chính xác của mọi điều kiện là 0,99+ trên 3.000 cặp, nên hầu hết
các cặp rơi vào ô đồng thuận và $m$ chỉ khoảng 11–20. Cell dưới tính p-value cho **kết quả cực đoan nhất có thể**
với $m$ đó — tức một model thắng toàn bộ các cặp bất đồng — rồi mô phỏng power của phép kiểm định.

Đây là **hiệu ứng chạm trần** (ceiling effect): phép đo quá dễ nên gần như mọi cặp đều đồng thuận, $m$ nhỏ, và
power gần bằng 0. Kết luận đúng là "phép đo không phân biệt được", **không phải** "hai model như nhau" — đúng
trường hợp thứ hai trong ba cách hiểu sai p-value ở mục 6.4.

In [ ]:
def p_mcnemar(n10, n01):
    m = n10 + n01
    if m == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(min(n10, n01) + 1)) / 2 ** m)


print(f"với m = {m_bd}, kết quả cực đoan nhất ({m_bd}-0) cho p = {p_mcnemar(m_bd, 0):.5f}")
print(f"kết quả thực tế ({n10}-{n01}) cho p = {p_mcnemar(n10, n01):.4f}\n")

rng = np.random.default_rng(5)
print("power của McNemar khi model A thật sự tốt hơn (mô phỏng 20.000 lần):")
print("  m  | A thắng 60% cặp bất đồng | A thắng 75% | A thắng 90%")
for m_gia in (m_bd, 50, 200):
    hang = []
    for pi in (0.6, 0.75, 0.9):
        thang = rng.binomial(m_gia, pi, size=20000)
        hang.append(float(np.mean([p_mcnemar(int(t), m_gia - int(t)) < 0.05 for t in thang])))
    print(f" {m_gia:3d} | {hang[0]:23.1%} | {hang[1]:11.1%} | {hang[2]:11.1%}")
print("\nVới m ~ 11 thì dù A tốt hơn hẳn, phép kiểm định vẫn gần như không bao giờ phát hiện được.")

## 10. Effect size và significance

### 10.1. Hai đại lượng độc lập

- **Effect size**: độ lớn hiệu ứng theo đơn vị bài toán, ở đây $\hat{\Delta}$ bpc, hoặc dạng tương đối
  $\hat{\Delta}/\mathrm{bpc}_B$.
- **Significance**: mức độ chắc chắn rằng hiệu ứng khác 0, đo bằng $z$ hoặc p-value.

Quan hệ giữa chúng, viết lại $z$ theo các đại lượng gốc:

$$z = \frac{\hat{\Delta}}{\mathrm{SE}} = \frac{\hat{\Delta}\sqrt{n}}{s_{A-B}}$$

Vì $z \propto \sqrt{n}$, **mọi hiệu ứng khác 0 đều trở thành có ý nghĩa thống kê khi $n$ đủ lớn**. Do đó câu "kết
quả có ý nghĩa thống kê" một mình không nói gì về tầm quan trọng thực tiễn; báo cáo phải đưa cả hai con số.

### 10.2. Hiệu ứng nhỏ nhất phát hiện được

Điều kiện để một phép kiểm định hai phía mức $\alpha$ đạt power $1-\beta$ với hiệu ứng thật $\Delta$:

$$\frac{|\Delta|}{\mathrm{SE}} \ \ge\ z_{1-\alpha/2} + z_{1-\beta}$$

**Ký hiệu mới:** $z_{1-\beta}$ — phân vị chuẩn ứng với power mong muốn (power 80% cho $z_{0{,}80} = 0{,}84$).

Giải thích: thống kê $z$ cần vượt ngưỡng $z_{1-\alpha/2}$ để bác bỏ; muốn nó vượt ngưỡng đó với xác suất
$1-\beta$, kỳ vọng của nó phải cách ngưỡng thêm $z_{1-\beta}$ độ lệch chuẩn nữa. (Công thức bỏ qua xác suất
$z$ vượt ngưỡng ở **đuôi ngược dấu**; phần đó nhỏ hơn $\alpha/2$ nhiều nên đây là xấp xỉ chuẩn trong sách.)

Với $\alpha = 0{,}05$ và power 80%: $z_{0{,}975} + z_{0{,}80} = 1{,}96 + 0{,}84 = 2{,}8$, nên
$\mathrm{MDE} = 2{,}8\,\mathrm{SE}$. Con số này dùng được cả theo chiều ngược: từ hiệu ứng muốn phát hiện, suy ra
$n$ cần thiết, theo $\mathrm{SE} \propto 1/\sqrt{n}$.

In [ ]:
print("hiệu ứng CỐ ĐỊNH, chỉ n thay đổi:")
print("     n   | SE         | z      | p")
for m in (50, 200, 1000, n, 20000):
    se_m = (a - b).std(ddof=1) / math.sqrt(m)
    zm = delta_hat / se_m
    print(f" {m:7d} | {se_m:.6f} | {zm:6.2f} | {2 * (1 - Phi(abs(zm))):.2e}")

z80 = phan_vi_chuan(0.80)
mde = (z95 + z80) * se_boot
print(f"\nz_0.975 + z_0.80 = {z95:.3f} + {z80:.3f} = {z95 + z80:.2f}")
print(f"MDE = {z95 + z80:.2f} × {se_boot:.6f} = {mde:.6f} bpc = {mde / bpc_b:.3%} tương đối")
print(f"hiệu ứng H1 quan sát được = {delta_hat:.6f} = {delta_hat / bpc_b:.3%} tương đối"
      f"  -> gấp {delta_hat / mde:.1f} lần MDE")

muc_tieu = 0.0005 * bpc_b
n_can = n * (se_boot / (muc_tieu / (z95 + z80))) ** 2
print(f"\nmuốn phát hiện hiệu ứng 0,05% ({muc_tieu:.6f} bpc) cần n ≈ {n_can:,.0f} văn bản")

## 11. Phân rã nguồn nhiễu — hạn chế thật của bảng kết quả

### 11.1. Mô hình

Viết giá trị đo được thành tổng các thành phần:

$$\hat{\Delta}_{\text{đo}} = \Delta_{\text{thật}} + \varepsilon_{\text{văn bản}} + \varepsilon_{\text{lần train}}$$

**Ký hiệu mới:** $\varepsilon$ — một sai số ngẫu nhiên kỳ vọng bằng 0, từ nguồn ghi ở chỉ số dưới.

Nếu các thành phần độc lập:

$$\operatorname{Var}(\hat{\Delta}_{\text{đo}}) = \operatorname{Var}(\varepsilon_{\text{văn bản}}) + \operatorname{Var}(\varepsilon_{\text{lần train}})$$

- $\varepsilon_{\text{văn bản}}$: do chọn tập test này thay vì tập khác. **Bootstrap ước lượng được** (mục 8).
- $\varepsilon_{\text{lần train}}$: do khởi tạo trọng số và tính bất định của fp16. **Bootstrap không thấy.**
  Muốn đo phải train lại — chính là lý do project chạy thêm seed 1.

### 11.2. Số đo được

Cell dưới lấy kết quả seed 1 (`results-v8`) và so **cùng một điều kiện, hai seed khác nhau**. Mọi chênh lệch ở
đó là nhiễu thuần tuý: hiệu ứng thật bằng 0 theo xây dựng.

In [ ]:
A1 = doc_load(RUNS / "results-v8" / "results" / "super-nfc_d8_s1.json")
B1 = doc_load(RUNS / "results-v8" / "results" / "bpe-nfc_d8_s1.json")

print("chênh lệch giữa hai seed của CÙNG một điều kiện (hiệu ứng thật = 0):")
nhieu_train = []
for ten, r0, r1 in [("bpe-nfc", B, B1), ("super-nfc", A, A1)]:
    ids = lay_giao([r0, r1])
    c = np.array([r0["docs"]["clean"]["chars"][i] for i in ids], float)
    d = bpc(nats_cua(r1, ids), c) - bpc(nats_cua(r0, ids), c)
    nhieu_train.append(abs(d))
    print(f"  {ten:10s}: seed1 - seed0 = {d:+.6f}")

print(f"\nsd(ε_văn bản)  từ bootstrap        = {se_boot:.6f}")
print(f"|ε_lần train| đo được (lớn nhất)   = {max(nhieu_train):.6f}")
print(f"=> nhiễu train lớn hơn nhiễu văn bản khoảng {max(nhieu_train) / se_boot:.1f} lần")
print(f"\nhiệu ứng H1 = {delta_hat:+.6f}, so với thanh nhiễu train {max(nhieu_train):.6f}"
      f" -> hiệu ứng lớn hơn {delta_hat / max(nhieu_train):.1f} lần")

### 11.3. Vì sao hai seed chỉ cho "thanh tham chiếu"

Với hai seed, bạn có **một** hiệu số cho mỗi điều kiện. Một quan sát không đủ để ước lượng độ lệch chuẩn: công
thức phương sai mẫu chia cho $n-1$, mà $n = 2$ cho đúng một bậc tự do, nên ước lượng cực kỳ không ổn định. Muốn
có $\mathrm{sd}(\varepsilon_{\text{lần train}})$ đáng tin cần 5–10 seed mỗi điều kiện.

Cell dưới cho thấy mức không ổn định đó là bao nhiêu: mô phỏng một thế giới có nhiễu train với độ lệch chuẩn biết
trước, rồi xem ước lượng từ 2 seed dao động thế nào so với ước lượng từ 10 seed.

Ngoài ra seed trong project **chỉ đổi khởi tạo trọng số**: `NANOCHAT_SEED` tác động vào `torch.manual_seed`, còn
dataloader của nanochat không xáo trộn dữ liệu. Nên thành phần nhiễu do **thứ tự dữ liệu** không nằm trong con số
đo được, và con số đó là **cận dưới** của nhiễu thật.

In [ ]:
rng = np.random.default_rng(6)
sd_that = 0.0005
for so_seed in (2, 5, 10, 30):
    uoc_luong = rng.normal(0, sd_that, size=(20000, so_seed)).std(axis=1, ddof=1)
    lo_q, hi_q = np.percentile(uoc_luong, [5, 95])
    print(f"{so_seed:3d} seed -> ước lượng sd nằm trong [{lo_q:.6f}, {hi_q:.6f}] 90% số lần"
          f"  (sd thật = {sd_that})")
print("\nVới 2 seed, ước lượng sd sai lệch tới vài lần -> chỉ dùng làm thanh tham chiếu, không làm số liệu.")

### 11.4. Áp dụng để đọc kết quả

- **H1**: hiệu ứng $0{,}0018$, nhiễu train $\le 0{,}0005$, và **cùng dấu ở cả hai seed**. Kết luận được.
- **H2 ở d8**: các hiệu ứng có độ lớn $0{,}0007$–$0{,}0051$, trong khi chênh lệch giữa hai seed của cùng một điều
  kiện trên văn bản bỏ dấu là $0{,}0015$–$0{,}0095$ (bốn hiệu số: hai điều kiện × hai biến thể). Bảng ghi
  `CI excludes 0 = yes` ở hai dòng vì p-value chỉ nhìn $\varepsilon_{\text{văn bản}}$, nhưng kết luận **không**
  đứng vững.

Đây là lý do mọi phát biểu trong báo cáo phải đối chiếu với thanh nhiễu seed, không chỉ nhìn cột khoảng tin cậy.
Cell dưới kiểm chứng trực tiếp: với từng biến thể văn bản, so nhiễu seed của **cả hai** điều kiện có seed 1 với
hiệu ứng H1 đo được trên biến thể đó. (H2 so `bpe-nfd` với `bpe-nfc`, mà `bpe-nfd` không có seed 1, nên thanh
nhiễu của H2 phải mượn từ các điều kiện có hai seed — thêm một lý do để coi nó là thanh tham chiếu thô.)

In [ ]:
for variant in ("clean", "strip50", "strip100"):
    nhieu_cac_dk = []
    for r0, r1 in ((B, B1), (A, A1)):                        # bpe-nfc và super-nfc, hai seed mỗi bên
        ids = lay_giao([r0, r1], variant)
        c = np.array([r0["docs"][variant]["chars"][i] for i in ids], float)
        nhieu_cac_dk.append(abs(bpc(nats_cua(r1, ids, variant), c) - bpc(nats_cua(r0, ids, variant), c)))
    nhieu = max(nhieu_cac_dk)

    ids2 = lay_giao([A, B], variant)
    c2 = np.array([A["docs"][variant]["chars"][i] for i in ids2], float)
    hieu_ung = bpc(nats_cua(A, ids2, variant), c2) - bpc(nats_cua(B, ids2, variant), c2)

    ket = "kết luận được" if abs(hieu_ung) > 2 * nhieu else "KHÔNG kết luận được (chìm trong nhiễu seed)"
    print(f"{variant:9s}: hiệu ứng H1 = {hieu_ung:+.6f} | nhiễu seed (bpe, super) = "
          f"({nhieu_cac_dk[0]:.4f}, {nhieu_cac_dk[1]:.4f}) -> {ket}")

### 11.5. Kiểm định không thua kém: cách H1 thật sự được chấm

H1 không hỏi "SuperBPE có **khác** BPE không" mà hỏi "SuperBPE có **tệ hơn quá 1%** không". Đó là một câu hỏi
khác, và cần một kiểm định khác.

**Định nghĩa.** Gọi $\delta = \mathrm{bpc}_A / \mathrm{bpc}_B - 1$ là chênh lệch tương đối và $m$ là **ngưỡng
chấp nhận được** chọn trước ($m = 1\%$ trong kế hoạch). Kiểm định không thua kém (non-inferiority) đảo vai hai giả
thuyết:

$$H_0:\ \delta \ge m \qquad H_1:\ \delta < m$$

Tức mặc định là "A tệ hơn quá ngưỡng", và dữ liệu phải chứng minh điều ngược lại.

**Công thức.** Theo đối ngẫu ở mục 6.2, bác bỏ $H_0$ ở mức $\alpha = 0{,}025$ một phía tương đương với

$$U_{95\%} < m$$

**Ký hiệu mới:** $U_{95\%}$ — cận trên của khoảng tin cậy 95% hai phía của $\delta$; $m$ — ngưỡng không thua kém.

**Khi nào áp dụng.** Khi phương án mới có lợi ích khác (ở đây: ít token hơn 18%, tức nhanh và rẻ hơn), và câu hỏi
là cái giá về chất lượng có nằm trong mức chấp nhận được không. Kiểm định "khác 0" trả lời sai câu hỏi đó: với đủ
nhiều văn bản, một chênh lệch 0,3% cũng "có ý nghĩa", dù nó nằm xa dưới ngưỡng 1%.

**Luật "resolved" của bảng.** Từ bản phân tích sửa ngày 26/09/2026, `results/summary.md` chỉ coi một so sánh là
đã phân giải khi cột `CI excludes 0` **và** cột `|Δ| > seed noise` đều là `yes`. Bảng dùng ngưỡng **một** lần nhiễu
seed; cell ở mục 11.4 dùng **hai** lần (chặt hơn). Với số liệu hiện có, hai ngưỡng chỉ khác nhau ở một dòng
(chi phí NFD của BPE ở d8) và cho cùng kết luận cho H1–H3.

Cell dưới chấm H1 bằng đúng hàm của project, ở cả ba cỡ:

In [ ]:
from vitok.analysis import load
from vitok.stats import paired_bootstrap_bpc

MARGIN = 0.01
all_runs = load(RUNS)
for depth in (6, 8, 10):
    ra, rb = all_runs[("super-nfc", depth, 0)], all_runs[("bpe-nfc", depth, 0)]
    ids = lay_giao([r for (c, dd, s), r in all_runs.items() if dd == depth])
    c = np.array([rb["docs"]["clean"]["chars"][i] for i in ids], float)
    res = paired_bootstrap_bpc(nats_cua(ra, ids), nats_cua(rb, ids), c)
    lo, hi = res["rel_ci95"]
    print(f"d{depth:<2d} super-nfc - bpe-nfc: delta = {res['rel_diff']:+.2%}, CI95 = [{lo:+.2%}, {hi:+.2%}]"
          f" | 'khác 0': {res['significant']} | không thua kém quá {MARGIN:.0%}: {hi < MARGIN}")

## 12. So sánh nhiều lần

Mỗi cỡ model, `vitok.analysis` chạy $m = 8$ so sánh. Nếu mỗi phép có xác suất dương tính giả $\alpha = 0{,}05$ và
các phép độc lập, xác suất có **ít nhất một** dương tính giả trong cả nhóm — gọi là **family-wise error rate** —
là

$$\mathrm{FWER} = 1 - (1-\alpha)^m = 1 - 0{,}95^8 = 0{,}337$$

**Lưu ý:** $m$ ở đây là **số phép kiểm định**, khác $m$ (số cặp bất đồng) ở mục 9.

Tức 34%. Hai cách xử lý:

- **Hiệu chỉnh Bonferroni**: dùng ngưỡng $\alpha/m$ cho mỗi phép. Bảo đảm $\mathrm{FWER} \le \alpha$ (theo bất
  đẳng thức Boole: xác suất *ít nhất một* sự kiện xảy ra không vượt tổng xác suất từng sự kiện — không cần giả
  định độc lập), nhưng bảo thủ nên làm giảm power.
- **Phân cấp bằng chứng**: chỉ định trước một phép là **chính**, các phép còn lại là phụ trợ hoặc khám phá và
  không dùng để tuyên bố.

Project theo cách thứ hai: kế hoạch phân tích (commit `4632328`, xem bằng
`git show 4632328:docs/analysis_plan.md`) xếp H1 là chính, H2 là phụ, phần còn lại là khám phá, và được
commit **trước khi train**.

Cần nói rõ trong báo cáo: đăng ký trước loại bỏ được một vấn đề khác — chọn phép kiểm định sau khi đã nhìn dữ
liệu, thường gọi là *p-hacking* hoặc *bậc tự do của người nghiên cứu* — nhưng **không** loại bỏ bài toán so sánh
nhiều lần. Hai vấn đề riêng biệt.

In [ ]:
rng = np.random.default_rng(7)
from vitok.analysis import COMPARISONS

R, m_test = 20000, len(COMPARISONS)   # số so sánh mỗi cỡ model, đọc thẳng từ mã phân tích
p_gia = rng.random((R, m_test))          # dưới H0, p-value phân phối đều trên [0,1]

print(f"mô phỏng {R} 'project', mỗi project chạy {m_test} kiểm định, H0 đúng ở cả {m_test}:")
print(f"  ít nhất một p < 0.05          : {(p_gia < 0.05).any(axis=1).mean():.1%}")
print(f"  công thức 1-(1-α)^m           : {1 - 0.95 ** m_test:.1%}")
print(f"  ít nhất một p < 0.05/{m_test} (Bonferroni): {(p_gia < 0.05 / m_test).any(axis=1).mean():.1%}")
print(f"\nngưỡng Bonferroni = 0.05/{m_test} = {0.05 / m_test:.4f}")
print(f"p-value của H1 ở d8 = {p:.2e} -> {'vẫn vượt ngưỡng' if p < 0.05 / m_test else 'không vượt'}")

## 13. Bẫy cần nhớ

1. Khoảng tin cậy chỉ bao phủ nguồn ngẫu nhiên được đưa vào mô phỏng; bootstrap theo văn bản **không** bao phủ
   nhiễu lần train (mục 11).
2. Ghép cặp chỉ hợp lệ khi hai bên được chấm trên đúng cùng tập; phải lấy giao trước khi so (mục 7.4).
3. "Không có ý nghĩa thống kê" không đồng nghĩa "không có hiệu ứng" — có thể chỉ là power thấp (mục 9).
4. Với $n$ lớn, hiệu ứng nhỏ về mặt thực tiễn vẫn cho p-value rất nhỏ. Luôn báo cáo effect size kèm theo (mục 10).
5. Phép kiểm định chọn sau khi nhìn dữ liệu không giữ được $\alpha$ như tuyên bố (mục 12).
6. p-value không phải xác suất $H_0$ đúng (mục 6.4).

## 14. Tóm tắt công thức

| Khái niệm | Công thức | Vai trò trong project |
|---|---|---|
| Sai số chuẩn của trung bình | $\mathrm{SE} = \sigma/\sqrt{n}$ | Nền của mọi khoảng tin cậy |
| Định lý giới hạn trung tâm | $(\bar X - \mu)/(\sigma/\sqrt n) \to \mathcal{N}(0,1)$ | Cho phép dùng phân vị chuẩn |
| Khoảng tin cậy | $\hat\theta \pm z_{1-\alpha/2}\,\mathrm{SE}$ | Cột `ci95` trong `summary.md` |
| p-value hai phía | $2(1-\Phi(\lvert z\rvert))$ | Cột `CI excludes 0` |
| Phương sai của hiệu | $\operatorname{Var}X + \operatorname{Var}Y - 2\operatorname{Cov}(X,Y)$ | Giải thích lợi ích 22 lần của ghép cặp |
| Bootstrap | $\mathrm{SE} \approx \mathrm{sd}(\hat\theta^{*(1)},\dots,\hat\theta^{*(B)})$ | `vitok.stats.paired_bootstrap_bpc` |
| McNemar exact | $p = \min(1, 2\sum_{i\le k}\binom{m}{i}2^{-m})$ | `vitok.stats.mcnemar_exact` |
| MDE | $(z_{1-\alpha/2} + z_{1-\beta})\,\mathrm{SE}$ | Ngưỡng hiệu ứng thiết kế phát hiện được |
| FWER | $1-(1-\alpha)^m$ | Lý do phải phân cấp H1/H2/khám phá |

## 15. Câu hỏi tự kiểm

1. Dẫn ra $\mathrm{SE}(\bar{X}) = \sigma/\sqrt{n}$ từ hai tính chất của phương sai.
2. Con số 1,96 đến từ đâu, và vì sao là phân vị $0{,}975$ chứ không phải $0{,}95$?
3. Dùng công thức phương sai của hiệu và $\rho$ đo được để giải thích tỷ lệ 22 lần giữa hai cách tính
   $\mathrm{SE}$.
4. Tập test tăng từ 1.996 lên 8.000 văn bản thì $\mathrm{SE}$ và $\mathrm{MDE}$ thay đổi thế nào?
5. Với $n_{10} = 3$, $n_{01} = 11$, tính p-value McNemar bằng tay rồi kiểm lại bằng hàm `p_mcnemar` ở mục 9.
6. Vì sao bootstrap phải rút **cả cặp** cho hai điều kiện cùng lúc?
7. Hiệu ứng H2 ở d8 có khoảng tin cậy không chứa 0, nhưng vẫn không kết luận. Viết lý do bằng ngôn ngữ phân rã
   nguồn nhiễu.
8. Nếu dùng Bonferroni cho 8 so sánh, ngưỡng p-value mới là bao nhiêu, và hiệu ứng H1 ở d8 có còn vượt ngưỡng?

**Đáp án gợi ý**

1. $\operatorname{Var}(\bar{X}) = \frac{1}{n^2}\sum_i \operatorname{Var}(X_i) = \frac{n\sigma^2}{n^2} = \frac{\sigma^2}{n}$, lấy căn bậc hai.
2. Là phân vị của chuẩn chuẩn tắc sao cho mỗi đuôi chứa 2,5%; tổng hai đuôi 5%, nên biên phải ở mức tích luỹ $1-0{,}05/2 = 0{,}975$.
3. $\operatorname{Var}(A-B) = \operatorname{Var}A + \operatorname{Var}B - 2\rho s_A s_B$; với $s_A \approx s_B$ mẫu số còn $\approx 2s^2(1-\rho)$, nên tỷ lệ độ lệch chuẩn là $1/\sqrt{1-\rho} \approx 22$.
4. $\mathrm{SE} \propto 1/\sqrt{n}$ nên giảm $\sqrt{8000/1996} = 2{,}0$ lần; $\mathrm{MDE} = 2{,}8\,\mathrm{SE}$ giảm đúng bằng đó.
5. $m = 14$, $k = 3$: $p = 2(1+14+91+364)/2^{14} = 0{,}0574$.
6. Vì tương quan $\rho \approx 0{,}998$ là thứ làm sai số chuẩn nhỏ đi 22 lần; rút độc lập phá tương quan đó và mô phỏng một phân phối lấy mẫu không tương ứng với thiết kế thực tế (đã chạy ở mục 8).
7. Khoảng tin cậy chỉ bao phủ $\varepsilon_{\text{văn bản}}$ ($\mathrm{sd} \approx 0{,}0003$–$0{,}0005$), trong khi hiệu ứng H2 ở d8 ($\le 0{,}0051$) không lớn hơn $|\varepsilon_{\text{lần train}}|$ đo được trên văn bản bỏ dấu ($0{,}0015$–$0{,}0095$). Nguồn nhiễu ràng buộc không được kiểm định bao phủ.
8. $0{,}05/8 = 0{,}00625$; $p \approx 10^{-8}$ nên vẫn vượt rất xa.

Cell cuối chạy lại câu 4 và câu 5 để đối chiếu.

In [ ]:
print("câu 4:")
se_8000 = se_boot * math.sqrt(n / 8000)
print(f"  SE : {se_boot:.6f} -> {se_8000:.6f}  (giảm {se_boot / se_8000:.2f} lần)")
print(f"  MDE: {mde:.6f} -> {(z95 + z80) * se_8000:.6f}")

print("\ncâu 5:")
print(f"  p_mcnemar(3, 11) = {p_mcnemar(3, 11):.4f}")

print("\ncâu 8:")
print(f"  ngưỡng Bonferroni = {0.05 / len(COMPARISONS):.4f}; p(H1) = {p:.2e} -> vượt: {p < 0.05 / len(COMPARISONS)}")

**Nguồn đọc thêm**

- [The Hitchhiker's Guide to Testing Statistical Significance in NLP](https://aclanthology.org/P18-1128/) — mục
  2–4: chọn kiểm định nào cho bài toán nào, gồm cả McNemar và bootstrap.
- [Statistical Significance Tests for MT Evaluation](https://aclanthology.org/W04-3250/) — bootstrap ghép cặp,
  6 trang, đúng phương pháp `vitok.stats` đang dùng.
- [With Little Power Comes Great Responsibility](https://arxiv.org/abs/2010.06595) — power và effect size trong
  NLP, mục 1–3.
- [Fine-Tuning Pretrained LMs: Weight Initializations, Data Orders, and Early Stopping](https://arxiv.org/abs/2002.06305)
  — nhiễu giữa các lần train, tương ứng mục 11 ở đây.
- [Reporting Score Distributions Makes a Difference](https://arxiv.org/abs/1707.09861) — vì sao báo cáo một con
  số từ một seed là sai lầm phổ biến.
- [Preregistering NLP Research](https://aclanthology.org/2021.naacl-main.51/) — mục 2–3, nền của
  kế hoạch phân tích đăng ký trước của project (commit `4632328`).